In [9]:
import os
import sys
import logging
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(project_root)
from config.conf import create_spark_session
from config.conf import config
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [10]:
spark = create_spark_session()

In [11]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [12]:
try:
    input_path = f"{config.root_path}{config.silver_zone}transfers"

    logger.info(f"Lecture du fichier transfers depuis : {input_path}")
    df_input = spark.read.option("header", "true").parquet(input_path)

    logger.info("✅ Lecture des données transfers terminée avec succès.\n")
except:
    logger.error(f"❌ Erreur lors de la lecture des données {input_path} : {e}", exc_info=True)

2025-05-19 09:14:14,362 - INFO - Lecture du fichier transfers depuis : s3a://loicverdier/silver/football_transfermarkt/transfers
2025-05-19 09:14:17,244 - INFO - ✅ Lecture des données transfers terminée avec succès.



In [13]:
df_input = df_input.filter(F.col("transfer_fee") > 0)
df_input.show()

+---------+-------------+---------------+------------+----------+--------------+--------------+------------+-------------------+--------------------+----------------+-------------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id|from_club_name|  to_club_name|transfer_fee|market_value_in_eur|         player_name|price_difference|transfer_year|
+---------+-------------+---------------+------------+----------+--------------+--------------+------------+-------------------+--------------------+----------------+-------------+
|  1138758|   2026-07-01|          26/27|         336|       631|   Sporting CP|       Chelsea|    52140000|           45000000|      Geovany Quenda|         7140000|         2026|
|   149729|   2025-07-01|          25/26|         294|       114|       Benfica|      Besiktas|     2000000|            2300000|          João Mário|         -300000|         2025|
|   234811|   2025-07-01|          25/26|         150|       336|    Real Betis|   Sporting CP|

In [22]:
df = df_input.groupBy("to_club_name", "transfer_year") \
    .agg(
        F.count("to_club_name").alias("nb_purchases"),
        (F.sum(F.col("transfer_fee"))).alias("total_purchases"),
        (F.sum(F.col("transfer_fee")) / F.count("to_club_name")).cast("int").alias("mean_purchases"),
        (F.sum(F.col("price_difference"))).alias("total_purchases_overprice"),
        (F.sum(F.col("price_difference")) / F.count("to_club_name")).cast("int").alias("mean_purchases_overprice")
        
    ) \
    .withColumnRenamed("to_club_name", "club_name")

sales = df_input.groupBy("from_club_name", "transfer_year") \
        .agg(
            F.count("from_club_name").alias("nb_sales"),
            (F.sum(F.col("transfer_fee")).alias("total_sales")),
            (F.sum(F.col("transfer_fee")) / F.count("from_club_name")).cast("int").alias("mean_sales"),
            (F.sum(F.col("price_difference"))).alias("total_sales_overprice"),
            (F.sum(F.col("price_difference")) / F.count("from_club_name")).cast("int").alias("mean_sales_overprice")
        ) \
        .withColumnRenamed("from_club_name", "club_name")


df = df.join(sales, on=["club_name", "transfer_year"], how="left") \
            .fillna(0) \
            .withColumn("balance", F.col("total_sales") - F.col("total_purchases"))

df = df.orderBy(F.col("total_purchases").desc())

df.show()

[Stage 79:=============================>                            (1 + 1) / 2]

+---------------+-------------+------------+---------------+--------------+-------------------------+------------------------+--------+-----------+----------+---------------------+--------------------+----------+
|      club_name|transfer_year|nb_purchases|total_purchases|mean_purchases|total_purchases_overprice|mean_purchases_overprice|nb_sales|total_sales|mean_sales|total_sales_overprice|mean_sales_overprice|   balance|
+---------------+-------------+------------+---------------+--------------+-------------------------+------------------------+--------+-----------+----------+---------------------+--------------------+----------+
|        Chelsea|         2023|          18|      782600000|      43477777|                281600000|                15644444|       7|  222800000|  31828571|            -30200000|            -4314285|-559800000|
|       Paris SG|         2023|           8|      349500000|      43687500|                 14500000|                 1812500|       6|   40000000| 

In [28]:
target_club = "Chelsea"
target_year = 2023

df_filtered = df_input.filter((F.col("to_club_name") == target_club) & (F.col("transfer_year") == target_year))

df_filtered.show()

df_filtered = df_input.filter((F.col("from_club_name") == target_club) & (F.col("transfer_year") == target_year))

df_filtered.show()

+---------+-------------+---------------+------------+----------+--------------+------------+------------+-------------------+------------------+----------------+-------------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id|from_club_name|to_club_name|transfer_fee|market_value_in_eur|       player_name|price_difference|transfer_year|
+---------+-------------+---------------+------------+----------+--------------+------------+------------+-------------------+------------------+----------------+-------------+
|   568177|   2023-09-01|          23/24|         281|       631|      Man City|     Chelsea|    47000000|           18000000|       Cole Palmer|        29000000|         2023|
|   465555|   2023-08-26|          23/24|         626|       631|   New England|     Chelsea|    16000000|            6000000|  Djordje Petrovic|        10000000|         2023|
|  1082850|   2023-08-24|          23/24|         221|       631|        Santos|     Chelsea|    16000000|         

+---------+-------------+---------------+------------+----------+--------------+------------+------------+-------------------+------------------+----------------+-------------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id|from_club_name|to_club_name|transfer_fee|market_value_in_eur|       player_name|price_difference|transfer_year|
+---------+-------------+---------------+------------+----------+--------------+------------+------------+-------------------+------------------+----------------+-------------+
|   392768|   2023-09-01|          23/24|         631|       703|       Chelsea|Nottm Forest|     3500000|           15000000|Callum Hudson-Odoi|       -11500000|         2023|
|   315779|   2023-07-13|          23/24|         631|         5|       Chelsea|    AC Milan|    20800000|           25000000| Christian Pulisic|        -4200000|         2023|
|   346483|   2023-07-05|          23/24|         631|       985|       Chelsea|     Man Utd|    64200000|         

In [24]:
# Trouver le club avec la meilleure balance pour chaque année
window = Window.partitionBy("transfer_year").orderBy(F.col("balance").desc())

best_balances = (
    df.withColumn("rank", F.row_number().over(window))
          .filter(F.col("rank") == 1)
          .drop("rank")
          .orderBy("transfer_year")
)

# Afficher les meilleurs clubs par année
best_balances.show()

+--------------+-------------+------------+---------------+--------------+-------------------------+------------------------+--------+-----------+----------+---------------------+--------------------+---------+
|     club_name|transfer_year|nb_purchases|total_purchases|mean_purchases|total_purchases_overprice|mean_purchases_overprice|nb_sales|total_sales|mean_sales|total_sales_overprice|mean_sales_overprice|  balance|
+--------------+-------------+------------+---------------+--------------+-------------------------+------------------------+--------+-----------+----------+---------------------+--------------------+---------+
|    Villarreal|         2002|           1|         750000|        750000|                        0|                       0|       0|          0|         0|                    0|                   0|  -750000|
| FC Bayern U17|         2004|           1|          15000|         15000|                        0|                       0|       0|          0|         0